In [ ]:
"""
Title: Python OOP Mini Project: A Simple Banking System
Author: Kacey Bates
Description: This project implements a simple banking system using Python's Object-Oriented Programming (OOP) principles. It includes classes for Bank, Account, and Customer, allowing users to create accounts, deposit and withdraw money, and view account details.
"""

# Customer records should have first name, last name, address, and customer id. 
class Customer: 
    def __init__(self, customer_id, first_name, last_name, address):
        self.customer_id = customer_id
        self.first_name = first_name
        self.last_name = last_name
        self.address = address


# Account records should have account type (checking, savings, etc), deposit, withdraw, and balance.
class Account:
    def __init__(self, account_id, account_type, initial_balance=0):
        self.account_id = account_id
        self.account_type = account_type
        self.balance = initial_balance

    def get_balance(self):
        return self.balance  

    def set_balance(self, amount):
        if amount >= 0:
            self.balance = amount
            return True
        return False

    def deposit(self, amount):
        if amount > 0:
            self.balance += amount
            return True
        return False

    def withdraw(self, amount):
        if 0 < amount <= self.balance:
            self.balance -= amount
            return True
        return False

# Services should include creating accounts, loans, credit cards, and other financial products.
class Service:
    def __init__(self, service_id, service_name):
        self.service_id = service_id
        self.service_name = service_name

    def get_service_details(self):
        return f"Service ID: {self.service_id}, Service Name: {self.service_name}"

    def loan_service(self, service_id, service_name, amount):
        # Placeholder for loan service logic
        return f"Loan # {self.service_id} of {amount} has been processed for customer."

    def credit_card_service(self, credit_limit):
        # Placeholder for credit card service logic
        return f"Credit card with a limit of {credit_limit} has been issued."

class Employee:
    def __init__(self, employee_id, first_name, last_name, position, salary):
        self.employee_id = employee_id
        self.first_name = first_name
        self.last_name = last_name
        self.position = position
        self.salary = salary

    def get_employee_details(self):
        return f"Employee ID: {self.employee_id}, Name: {self.first_name} {self.last_name}, Position: {self.position}, Salary: {self.salary}"


In [2]:
if "Customer" not in globals() or "Account" not in globals():
    raise RuntimeError("Run the first notebook cell that defines Customer and Account before running this database-backed section.")

import sqlite3
from pathlib import Path

DB_PATH = Path("bank.db")

def init_db(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        customer_id TEXT PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        address TEXT
    )
    """)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS accounts (
        account_id TEXT PRIMARY KEY,
        customer_id TEXT NOT NULL,
        account_type TEXT NOT NULL,
        balance REAL NOT NULL DEFAULT 0,
        FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
    )
    """)
    conn.commit()
    conn.close()


def save_customer(customer, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
    INSERT OR REPLACE INTO customers (customer_id, first_name, last_name, address)
    VALUES (?, ?, ?, ?)
    """, (customer.customer_id, customer.first_name, customer.last_name, customer.address))
    conn.commit()
    conn.close()


def save_account(account, customer_id, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
    INSERT OR REPLACE INTO accounts (account_id, customer_id, account_type, balance)
    VALUES (?, ?, ?, ?)
    """, (account.account_id, customer_id, account.account_type, account.balance))
    conn.commit()
    conn.close()


def get_customer(customer_id, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT customer_id, first_name, last_name, address FROM customers WHERE customer_id = ?", (customer_id,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return Customer(*row)
    return None


def get_accounts_for_customer(customer_id, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT account_id, account_type, balance FROM accounts WHERE customer_id = ?", (customer_id,))
    rows = cursor.fetchall()
    conn.close()
    return [Account(account_id, account_type, balance) for account_id, account_type, balance in rows]


class Bank:
    def __init__(self, db_path=DB_PATH):
        self.db_path = db_path
        init_db(self.db_path)

    def create_customer(self, customer_id, first_name, last_name, address):
        customer = Customer(customer_id, first_name, last_name, address)
        save_customer(customer, self.db_path)
        return customer

    def create_account(self, account_id, customer_id, account_type, initial_balance=0):
        customer = get_customer(customer_id, self.db_path)
        if not customer:
            raise ValueError(f"No customer found with id {customer_id}")
        account = Account(account_id, account_type, initial_balance)
        save_account(account, customer_id, self.db_path)
        return account

    def deposit(self, account_id, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT balance FROM accounts WHERE account_id = ?", (account_id,))
        row = cursor.fetchone()
        if not row:
            conn.close()
            raise ValueError(f"No account with id {account_id}")
        balance = row[0] + amount
        cursor.execute("UPDATE accounts SET balance = ? WHERE account_id = ?", (balance, account_id))
        conn.commit()
        conn.close()
        return balance

    def withdraw(self, account_id, amount):
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT balance FROM accounts WHERE account_id = ?", (account_id,))
        row = cursor.fetchone()
        if not row:
            conn.close()
            raise ValueError(f"No account with id {account_id}")
        balance = row[0]
        if amount > balance:
            conn.close()
            raise ValueError("Insufficient funds")
        balance -= amount
        cursor.execute("UPDATE accounts SET balance = ? WHERE account_id = ?", (balance, account_id))
        conn.commit()
        conn.close()
        return balance

    def list_customers(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT customer_id, first_name, last_name, address FROM customers")
        rows = cursor.fetchall()
        conn.close()
        return [Customer(*row) for row in rows]

    def list_accounts(self, customer_id=None):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        if customer_id:
            cursor.execute("SELECT account_id, account_type, balance FROM accounts WHERE customer_id = ?", (customer_id,))
        else:
            cursor.execute("SELECT account_id, account_type, balance FROM accounts")
        rows = cursor.fetchall()
        conn.close()
        return [Account(account_id, account_type, balance) for account_id, account_type, balance in rows]

In [ ]:
# Example bank flow: create customer, create account, deposit, withdraw, and list saved data.
bank = Bank()

customer = bank.create_customer("C001", "Kacey", "Bates", "123 Main St")
print("Created customer:", customer.customer_id, customer.first_name, customer.last_name)

account = bank.create_account("A001", customer.customer_id, "checking", 100)
print("Created account:", account.account_id, account.account_type, account.balance)

balance_after_deposit = bank.deposit("A001", 50)
print("Balance after deposit:", balance_after_deposit)

balance_after_withdrawal = bank.withdraw("A001", 30)
print("Balance after withdrawal:", balance_after_withdrawal)

print("All customers:")
for saved_customer in bank.list_customers():
    print(saved_customer.customer_id, saved_customer.first_name, saved_customer.last_name, saved_customer.address)

print("Accounts for C001:")
for saved_account in bank.list_accounts(customer.customer_id):
    print(saved_account.account_id, saved_account.account_type, saved_account.balance)

Created customer: C001 Kacey Bates
Created account: A001 checking 100
Balance after deposit: 150.0
Balance after withdrawal: 120.0
All customers:
C001 Kacey Bates 123 Main St
Accounts for C001:
A001 checking 120.0


In [4]:
# open bank db as a sqlite3 connection and print out the customers and accounts tables to verify data is saved correctly:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT * FROM customers")
customers = cursor.fetchall()
print("Customers table:")
for customer in customers:
    print(customer)

cursor.execute("SELECT * FROM accounts")
accounts = cursor.fetchall()
print("Accounts table:")
for account in accounts:
    print(account)
conn.close()



Customers table:
('C001', 'Kacey', 'Bates', '123 Main St')
Accounts table:
('A001', 'C001', 'checking', 120.0)
